# VAE curvature experiments on Colab GPU

**Before running:** Runtime -> Change runtime type -> Hardware accelerator -> **T4 GPU** (or A100/L4).

Required files in your project folder:

```
datasets.py  vaediagnostic.py  curvature_plot.py
new_tor_vae_charts.py  new_sph_vae_chart.py
bin_scripts/sphere_data.py        <-- easy to forget, it is not optional
```


## 1. Check the GPU is actually attached

In [1]:
!nvidia-smi
import torch
print("torch", torch.__version__, "| cuda available:", torch.cuda.is_available())
print("device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")


Tue Aug 11 11:15:37 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8             14W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Get your code into the runtime

Pick **one** of the three options below. Drive is the one to use if you want the
PDFs to survive the session dying.

In [2]:
# --- Option A: Google Drive (recommended) -------------------------------
from google.colab import drive
drive.mount('/content/drive')

PROJECT = '/content/drive/MyDrive/Bachelorarbeit/projects'   # <-- folder holding your .py files
%cd $PROJECT
!ls


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/Bachelorarbeit/projects
bin_scripts	       __pycache__	    standardvae.py
curvature_plot.py      runstore.py	    trial
datasets.py	       sandbox_log.py	    vaediagnostic.py
new_sph_vae_chart.py   sandbox.py	    vaediagnostics_3d.py
new_tor_vae_charts.py  standardvae_3dim.py  vis_datasets.py


In [3]:
# --- Option B: clone from GitHub ---------------------------------------
# !git clone https://github.com/<you>/<repo>.git /content/vae_project
# PROJECT = '/content/vae_project'
# %cd $PROJECT

# --- Option C: upload a zip -------------------------------------------
# from google.colab import files
# files.upload()          # pick your project.zip
# !unzip -o project.zip -d /content/vae_project
# PROJECT = '/content/vae_project'
# %cd $PROJECT


## 3. Make imports work

`from bin_scripts.sphere_data import ...` only resolves if the *parent* of
`bin_scripts/` is on `sys.path`. `%cd` into the project root handles that.

In [4]:
import os, sys
sys.path.insert(0, os.getcwd())

import numpy as np, torch, matplotlib.pyplot as plt

from datasets import make_torus_dataset, make_plane_dataset, make_plane_dataset_old
from vaediagnostic import diagnose, diagnose_sphere, plot_abs_curvature_sphere
from new_tor_vae_charts import TorusVAE as tnewVAE, train as ttrain
from new_sph_vae_chart import SphereVAE as snewVAE, train as strain
from curvature_plot import plot_abs_curvature

try:
    from bin_scripts.sphere_data import make_sphere_dataset_try3
except ModuleNotFoundError as e:
    print("!! bin_scripts/sphere_data.py missing:", e)
    from datasets import make_sphere_dataset_new as make_sphere_dataset_try3
    print("   falling back to make_sphere_dataset_new -- NOT the same dataset")

print("imports ok")


imports ok


## 4. Config

`CHUNK` is the important one. The curvature code vmaps a triple-nested
`jacfwd` over grid points; at `chunk=32` on a GPU you launch thousands of
tiny kernels and spend all your time on launch overhead. 512-2048 is a far
better fit and the memory cost is still small.

Note the curvature pass runs in **float64**. On a T4, FP64 throughput is 1/32
of FP32, so this part is not where the GPU shines -- it is still faster than
Colab's 2-core CPU, but do not expect a 10x speedup there. The training loop
is where the GPU pays off.

In [5]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CHUNK  = 1024 if DEVICE == "cuda" else 32
GRID   = 100          # torus grid (grid x grid)
NTHETA, NPHI = 64, 128    # sphere grid
EPOCHS = 1000

# Write outputs to Drive so a disconnect does not cost you the run.
OUT = '/content/drive/MyDrive/vae_results/11_08_26_topo'
os.makedirs(OUT, exist_ok=True)
LOG = os.path.join(OUT, 'curvature_log.txt')

print(f"device={DEVICE}  chunk={CHUNK}  out={OUT}")


device=cuda  chunk=1024  out=/content/drive/MyDrive/vae_results/11_08_26_topo


## 5. The driver

Same six experiments, same figure names. Differences from your script:

- `device=DEVICE` passed to `ttrain`/`strain` (explicit rather than implicit)
- `device=DEVICE, chunk=CHUNK` passed to the curvature plotters -- **this is
  the change that actually puts the curvature pass on the GPU**
- output paths point at Drive
- `plt.close('all')` after each block; six runs of unclosed figures trips
  matplotlib's 20-figure warning and leaks memory
- encoding is batched, and CUDA memory is released between runs
- each run is timed, so you can see where the wall clock goes

In [6]:
import time, gc

@torch.no_grad()
def encode_mu(model, data, batch=4096):
    model.eval()
    return torch.cat([model.encode(data[i:i+batch])[0]
                      for i in range(0, data.shape[0], batch)], dim=0)

def free():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def run_torus(key, data_fn, label):
    t0 = time.time()
    print(f"\n=== {key}  ({label})")
    val_x, val_ang = make_val(data_fn)
    train_x, _ = data_fn()
    train_x = torch.as_tensor(train_x, dtype=torch.float32, device=DEVICE)
    train_x_of[key] = train_x
    curve = []
    model, data, angles, history = ttrain(epochs=EPOCHS, data_fn=data_fn,
                                          model_cls=tnewVAE, device=DEVICE,
                                          eval_hook=make_hook(curve, train_x, val_x),
                                          eval_every=max(1, EPOCHS // 50))
    t_train = time.time() - t0

    diagnose(model, data, angles, save=os.path.join(OUT, f"latent_{key}.pdf"))
    plt.close('all')
    plot_curve(curve, key)
    plot_worst_points(model, key, "torus", angles, val_x, val_ang)
    model = model.double()
    mu = encode_mu(model, data.double())

    fig, K, V = plot_abs_curvature(model, grid=GRID, chunk=CHUNK, device=DEVICE,
                                   mu=mu,
                                   fname=os.path.join(OUT, f"curvature_{key}.pdf"),
                                   log_path=LOG, log_tag=key)
    plt.close('all')
    np.savez_compressed(os.path.join(OUT, f"arrays_{key}.npz"),
                        K=K, V=V, mu=mu.detach().cpu().numpy())
    print(f"{key} done  (train {t_train:.0f}s, total {time.time()-t0:.0f}s)")
    del model, data, mu; free()

def run_sphere(key, data_fn, label):
    t0 = time.time()
    print(f"\n=== {key}  ({label})")
    val_x, val_ang = make_val(data_fn)
    train_x, _ = data_fn()
    train_x = torch.as_tensor(train_x, dtype=torch.float32, device=DEVICE)
    train_x_of[key] = train_x
    curve = []
    model, data, angles = strain(epochs=EPOCHS, data_fn=data_fn,
                                          model_cls=snewVAE, device=DEVICE,
                                          eval_hook=make_hook(curve, train_x, val_x),
                                          eval_every=max(1, EPOCHS // 50))
    t_train = time.time() - t0

    diagnose_sphere(model, data, angles, save=os.path.join(OUT, f"latent_{key}.pdf"))
    plt.close('all')
    plot_curve(curve, key)
    plot_worst_points(model, key, "sphere", angles, val_x, val_ang)
    model = model.double()
    mu = encode_mu(model, data.double())

    fig, K, V = plot_abs_curvature_sphere(model, n_theta=NTHETA, n_phi=NPHI,
                                          chunk=CHUNK, device=DEVICE, mu=mu,
                                          fname=os.path.join(OUT, f"curvature_{key}.pdf"),
                                          log_path=LOG, log_tag=key)
    plt.close('all')
    np.savez_compressed(os.path.join(OUT, f"arrays_{key}.npz"),
                        K=K, V=V, mu=mu.detach().cpu().numpy())
    print(f"{key} done  (train {t_train:.0f}s, total {time.time()-t0:.0f}s)")
    del model, data, mu; free()


In [7]:
import pandas as pd
from new_sph_vae_chart import angles_from_vec

VAL_N, VAL_SEED, N_WORST = 2000, 1234, 500

def _np(a):
    return a.detach().cpu().numpy() if isinstance(a, torch.Tensor) else np.asarray(a)

@torch.no_grad()
def _recon(model, x):
    mu = model.encode(x)[0]
    # SphereVAE.encode returns a unit 3-vector; decode() wants 2 angles,
    # decode_vec() is the matching pair. TorusVAE.encode returns angles.
    return model.decode_vec(mu) if hasattr(model, "decode_vec") else model.decode(mu)

@torch.no_grad()
def per_sample_err(model, x, batch=4096):
    model.eval()
    p = next(model.parameters())
    x = x.to(device=p.device, dtype=p.dtype)
    e = torch.cat([((x[i:i+batch] - _recon(model, x[i:i+batch]))**2).sum(1)
                   for i in range(0, x.shape[0], batch)])
    model.train()
    return _np(e)

def make_val(data_fn, device=None):
    imgs, ang = data_fn(n=VAL_N, seed=VAL_SEED)
    return (torch.as_tensor(imgs, dtype=torch.float32, device=device or DEVICE),
            np.asarray(ang))

def make_hook(curve, train_x, val_x):
    def hook(model, ep):
        e_tr, e_va = per_sample_err(model, train_x), per_sample_err(model, val_x)
        curve.append(dict(epoch=ep, train=e_tr.mean(), val=e_va.mean(),
                          val_median=np.median(e_va)))
    return hook

def plot_curve(curve, key, panel_labels=("a)", "b)"), label_size=14):
    c = pd.DataFrame(curve)
    fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
    ax[0].plot(c.epoch, c.train, color="0.3", label="train")
    ax[0].plot(c.epoch, c.val, color="crimson", label="validation mean")
    ax[0].plot(c.epoch, c.val_median, "--", color="crimson", alpha=0.6,
               label="validation median")
    ax[0].set_yscale("log"); ax[0].set_xlabel("epoch")
    ax[0].set_ylabel("reconstruction error"); ax[0].legend(fontsize=8)

    ax[1].plot(c.epoch, c.val / c.train, color="0.3", label="validation / train")
    ax[1].plot(c.epoch, c.val / c.val_median, color="steelblue",
               label="mean / median (validation)")
    ax[1].axhline(1.0, color="0.7", lw=1)
    ax[1].set_xlabel("epoch"); ax[1].set_ylabel("ratio"); ax[1].legend(fontsize=8)

    for a, lab in zip(ax, panel_labels):
        a.text(-0.12, 1.02, lab, transform=a.transAxes, fontsize=label_size,
               fontweight="bold", va="bottom", ha="left")
    fig.tight_layout()
    fig.savefig(os.path.join(OUT, f"curve_{key}.pdf"), dpi=150, bbox_inches="tight")
    c.to_csv(os.path.join(OUT, f"curve_{key}.csv"), index=False)
    plt.close(fig)
    return c

def plot_worst_points(model, key, kind, train_ang, val_x, val_ang,
                      n_worst=N_WORST, n_points=4000, s=4, cyclic=True,
                      gt_shift=None, panel_labels=("a)", "b)"), label_size=14):
    """a) latent codes coloured by true gamma_1, worst held-out points in black
       b) the same worst points in ground-truth coordinates"""
    e_va = per_sample_err(model, val_x)
    bad = np.argsort(e_va)[-n_worst:]

    with torch.no_grad():
        p = next(model.parameters())
        mu_bg = model.encode(train_x_of[key].to(device=p.device, dtype=p.dtype))[0]
        mu_bad = model.encode(val_x[bad].to(device=p.device, dtype=p.dtype))[0]
        if kind == "sphere":                      # unit 3-vector -> (theta, phi)
            mu_bg, mu_bad = angles_from_vec(mu_bg), angles_from_vec(mu_bad)
    mu_bg, mu_bad = _np(mu_bg), _np(mu_bad)

    true = np.asarray(train_ang); va = np.asarray(val_ang).copy()
    if gt_shift is not None:
        va = np.mod(va + gt_shift, 2 * np.pi)

    n = min(n_points, mu_bg.shape[0])
    idx = np.random.default_rng(0).choice(mu_bg.shape[0], size=n, replace=False)
    mu_bg, true = mu_bg[idx], true[idx]

    if cyclic:
        cbg, cmap, vmin, vmax = np.mod(true[:, 0], 2*np.pi), "hsv", 0.0, 2*np.pi
    else:
        cbg, cmap, vmin, vmax = true[:, 0], "viridis", None, None

    fig, ax = plt.subplots(1, 2, figsize=(12, 5.5))
    sc = ax[0].scatter(mu_bg[:, 0], mu_bg[:, 1], s=s, c=cbg, cmap=cmap,
                       vmin=vmin, vmax=vmax, linewidths=0, alpha=0.75)
    ax[0].scatter(mu_bad[:, 0], mu_bad[:, 1], s=3*s, c="black", linewidths=0,
                  label=f"worst {n_worst} validation")
    if kind == "torus":
        ax[0].set_xlim(-np.pi, np.pi); ax[0].set_ylim(-np.pi, np.pi)
        ax[0].set_xticks([-3,-2,-1,0,1,2,3]); ax[0].set_yticks([-3,-2,-1,0,1,2,3])
        ax[0].set_xlabel(r"$\theta_1$"); ax[0].set_ylabel(r"$\theta_2$")
    else:
        ax[0].set_xlim(0, np.pi); ax[0].set_ylim(-np.pi, np.pi)
        ax[0].set_xlabel(r"$\theta$ (polar)"); ax[0].set_ylabel(r"$\phi$ (azimuth)")
    ax[0].set_aspect("equal"); ax[0].legend(fontsize=8)
    fig.colorbar(sc, ax=ax[0], fraction=0.046, pad=0.02, label=r"true $\gamma_1$")

    ax[1].scatter(va[:, 0], va[:, 1], s=2, c="0.85", linewidths=0)
    ax[1].scatter(va[bad, 0], va[bad, 1], s=3*s, c="black", linewidths=0)
    ax[1].set_aspect("equal")
    ax[1].set_xlabel(r"true $\gamma_1$"); ax[1].set_ylabel(r"true $\gamma_2$")

    for a, lab in zip(ax, panel_labels):
        a.text(-0.12, 1.02, lab, transform=a.transAxes, fontsize=label_size,
               fontweight="bold", va="bottom", ha="left")
    fig.tight_layout()
    fig.savefig(os.path.join(OUT, f"worst_{key}.pdf"), dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  worst {n_worst} carry {100*e_va[bad].sum()/e_va.sum():.1f}% of total error")
    return e_va, bad

train_x_of = {}

## 6. Run

One cell per experiment, deliberately. If the runtime drops halfway through
you only lose the current experiment, not all six.

In [9]:
run_torus("border_flat_torus", make_plane_dataset,     "bordered flat plane -> T2")



=== border_flat_torus  (bordered flat plane -> T2)
epoch    0  loss=717.771  recon=717.660  kl=0.111  sigma_floor=1.000  sigma_x=0.961
epoch   50  loss=-448.328  recon=-449.957  kl=1.629  sigma_floor=0.676  sigma_x=0.141
epoch  100  loss=-615.987  recon=-618.401  kl=2.414  sigma_floor=0.457  sigma_x=0.111
epoch  150  loss=-804.641  recon=-807.813  kl=3.172  sigma_floor=0.309  sigma_x=0.087
epoch  200  loss=-1033.580  recon=-1037.556  kl=3.976  sigma_floor=0.209  sigma_x=0.065
epoch  250  loss=-1261.076  recon=-1265.828  kl=4.751  sigma_floor=0.141  sigma_x=0.048
epoch  300  loss=-1481.196  recon=-1486.726  kl=5.530  sigma_floor=0.096  sigma_x=0.036
epoch  350  loss=-1678.521  recon=-1684.840  kl=6.319  sigma_floor=0.065  sigma_x=0.028
epoch  400  loss=-1881.265  recon=-1888.373  kl=7.107  sigma_floor=0.044  sigma_x=0.022
epoch  450  loss=-2037.088  recon=-2044.976  kl=7.888  sigma_floor=0.030  sigma_x=0.018
epoch  500  loss=-2196.636  recon=-2205.298  kl=8.662  sigma_floor=0.020  sigm

In [10]:
run_torus("flat_torus",        make_plane_dataset_old, "flat plane -> T2")


=== flat_torus  (flat plane -> T2)
epoch    0  loss=713.671  recon=713.530  kl=0.140  sigma_floor=1.000  sigma_x=0.961
epoch   50  loss=-503.194  recon=-503.194  kl=0.000  sigma_floor=0.676  sigma_x=0.131
epoch  100  loss=-509.321  recon=-509.519  kl=0.198  sigma_floor=0.457  sigma_x=0.127
epoch  150  loss=-1398.754  recon=-1401.926  kl=3.172  sigma_floor=0.309  sigma_x=0.041
epoch  200  loss=-1697.383  recon=-1701.359  kl=3.976  sigma_floor=0.209  sigma_x=0.028
epoch  250  loss=-2037.198  recon=-2041.950  kl=4.751  sigma_floor=0.141  sigma_x=0.018
epoch  300  loss=-2328.383  recon=-2333.913  kl=5.530  sigma_floor=0.096  sigma_x=0.012
epoch  350  loss=-2599.547  recon=-2605.866  kl=6.319  sigma_floor=0.065  sigma_x=0.009
epoch  400  loss=-2849.026  recon=-2856.133  kl=7.107  sigma_floor=0.044  sigma_x=0.006
epoch  450  loss=-3085.085  recon=-3092.973  kl=7.888  sigma_floor=0.030  sigma_x=0.005
epoch  500  loss=-3302.189  recon=-3310.851  kl=8.662  sigma_floor=0.020  sigma_x=0.004
epoc

In [11]:
run_torus("torus_torus",       make_torus_dataset,     "torus -> T2")


=== torus_torus  (torus -> T2)
epoch    0  loss=718.994  recon=718.900  kl=0.094  sigma_floor=1.000  sigma_x=0.961
epoch   50  loss=-438.878  recon=-440.508  kl=1.629  sigma_floor=0.676  sigma_x=0.143
epoch  100  loss=-627.774  recon=-630.188  kl=2.414  sigma_floor=0.457  sigma_x=0.109
epoch  150  loss=-844.932  recon=-848.104  kl=3.172  sigma_floor=0.309  sigma_x=0.082
epoch  200  loss=-1107.330  recon=-1111.306  kl=3.976  sigma_floor=0.209  sigma_x=0.059
epoch  250  loss=-1371.452  recon=-1376.203  kl=4.751  sigma_floor=0.141  sigma_x=0.042
epoch  300  loss=-1643.074  recon=-1648.604  kl=5.530  sigma_floor=0.096  sigma_x=0.030
epoch  350  loss=-1912.723  recon=-1919.042  kl=6.319  sigma_floor=0.065  sigma_x=0.021
epoch  400  loss=-2169.865  recon=-2176.973  kl=7.107  sigma_floor=0.044  sigma_x=0.015
epoch  450  loss=-2399.362  recon=-2407.250  kl=7.888  sigma_floor=0.030  sigma_x=0.011
epoch  500  loss=-2617.858  recon=-2626.519  kl=8.662  sigma_floor=0.020  sigma_x=0.008
epoch  550

In [12]:
run_sphere("border_flat_sphere", make_plane_dataset,       "bordered flat plane -> S2")


=== border_flat_sphere  (bordered flat plane -> S2)
  worst 500 carry 45.4% of total error
|K|          : median 3.424e-05  max 1.992e-03
sqrt(det G)  : median 2.032e+04  max 7.131e+04
K > 0 on     : 69.5% of the grid
Fisher area  : 3.760384e+05
int K dA     : 12.426868   target 4pi = 12.566371   deviation 1.11%
int |K| dA   : 33.107466   (>= |int K dA|; equal iff K has one sign)
[eps = 0.08: excluded caps carry ~2pi*eps^2 = 0.0402 of curvature for a round metric]
border_flat_sphere done  (train 244s, total 259s)


In [13]:
run_sphere("flat_sphere",        make_plane_dataset_old,   "flat plane -> S2")


=== flat_sphere  (flat plane -> S2)
  worst 500 carry 56.8% of total error
|K|          : median 9.406e-06  max 4.715e-03
sqrt(det G)  : median 2.277e+04  max 8.688e+05
K > 0 on     : 55.0% of the grid
Fisher area  : 6.258544e+05
int K dA     : 12.583512   target 4pi = 12.566371   deviation 0.14%
int |K| dA   : 21.932605   (>= |int K dA|; equal iff K has one sign)
[eps = 0.08: excluded caps carry ~2pi*eps^2 = 0.0402 of curvature for a round metric]
flat_sphere done  (train 243s, total 260s)


In [14]:
EPOCHS = 500
run_sphere("sphere_sphere",      make_sphere_dataset_try3, "sphere -> S2")


=== sphere_sphere  (sphere -> S2)
  worst 500 carry 49.0% of total error
|K|          : median 6.608e-05  max 1.398e-02
sqrt(det G)  : median 1.817e+04  max 4.852e+04
K > 0 on     : 67.6% of the grid
Fisher area  : 3.615030e+05
int K dA     : 12.591006   target 4pi = 12.566371   deviation 0.20%
int |K| dA   : 41.524698   (>= |int K dA|; equal iff K has one sign)
[eps = 0.08: excluded caps carry ~2pi*eps^2 = 0.0402 of curvature for a round metric]
sphere_sphere done  (train 124s, total 140s)


## 7. Collect results

Skip this if you wrote straight to Drive.

## Optional: make training genuinely fast

At `batch_size=256` over 10000 samples you run 40 optimiser steps per epoch on
a network with 256-unit hidden layers. That is far too small to saturate any
GPU -- you are paying kernel-launch latency, not compute. Raising the batch
size is the single biggest speedup available:

```python
model, data, angles, history = ttrain(epochs=EPOCHS, batch_size=2048, lr=2e-3,
                                      data_fn=data_fn, model_cls=tnewVAE,
                                      device=DEVICE)
```

**This changes the optimisation, not just the speed.** Fewer, larger steps with
the same `lr` means less total progress and less gradient noise, and the sigma
floor anneal in `ttrain` is indexed by epoch, so its schedule relative to the
number of steps shifts. Whether the encoder settles into the same winding
number is exactly the thing this experiment is measuring. Validate on one
experiment against your CPU baseline before adopting it everywhere.

The same caveat applies to GPU vs CPU generally: `torch.manual_seed(42)` inside
`train` does not make CUDA and CPU produce identical trajectories. If figures
already in your thesis came from CPU runs, expect the reruns to differ in
detail -- and if a conclusion flips, that tells you something about how tightly
these runs are determined, which is worth knowing either way.
